# Grokking replication: synthetic modular addition

Task: `(a, b) -> (a + b) mod p`, one-hot(a) concat one-hot(b) as input, `p`-way classification, **clean labels** (no noise).

One-hot inputs are mutually orthogonal for every distinct `(a, b)` pair, so the network can only generalize by discovering the additive structure from scratch during training. With strong weight decay (`AdamW`, `weight_decay=1.0`), this produces the grokking signature: training accuracy saturates to ~100% almost immediately, while test accuracy lags near chance for a long stretch, then rises sharply to ~100% much later in training.

**Before running**: set `Runtime > Change runtime type > GPU` so this actually trains on GPU.

In [ ]:
import csv
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={device}")

In [ ]:
# --- dataset ---
P = 97                  # modulus / number of classes (matches Power et al. 2022)
TRAIN_FRACTION = 0.5    # fraction of the p*p domain used for training
DATA_SEED = 0

# --- model / optimization ---
HIDDEN = 128
SEED = 0
LR = 1e-3
WEIGHT_DECAY = 1.0      # strong weight decay is what makes grokking tractable
MAX_EPOCHS = 100_000
LOG_EVERY = 50
GROK_PATIENCE_CHECKS = 20    # keep logging this many more checkpoints after grokking, then stop
GROK_THRESHOLD = 0.99

In [ ]:
def make_dataset(p=P, train_frac=TRAIN_FRACTION, seed=DATA_SEED, device="cpu"):
    g = torch.Generator().manual_seed(seed)

    a_all = torch.arange(p * p) // p
    b_all = torch.arange(p * p) % p
    labels_all = (a_all + b_all) % p

    def onehot_pairs(a, b, p):
        n = a.shape[0]
        X = torch.zeros(n, 2 * p)
        X[torch.arange(n), a] = 1.0
        X[torch.arange(n), p + b] = 1.0
        return X

    train_size = round(train_frac * p * p)
    perm = torch.randperm(p * p, generator=g)
    train_idx, test_idx = perm[:train_size], perm[train_size:]

    X_all = onehot_pairs(a_all, b_all, p)
    X_train, y_train = X_all[train_idx].to(device), labels_all[train_idx].to(device)
    X_test, y_test = X_all[test_idx].to(device), labels_all[test_idx].to(device)
    return X_train, y_train, X_test, y_test


class OneLayerMLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)


def accuracy(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()


def weight_norm(model):
    """L2 norm across every parameter, flattened -- the standard grokking-paper
    diagnostic for watching weight decay shrink the solution during the lag
    between memorization and generalization."""
    return torch.sqrt(sum(p.pow(2).sum() for p in model.parameters())).item()

In [ ]:
X_train, y_train, X_test, y_test = make_dataset(device=device)
print(f"p={P}, train_size={X_train.shape[0]}, test_size={X_test.shape[0]}")

torch.manual_seed(SEED)
model = OneLayerMLP(X_train.shape[1], HIDDEN, P).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.CrossEntropyLoss()

log = []  # (epoch, train_acc, test_acc, train_loss, weight_norm)
grok_checks_remaining = None

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    opt.zero_grad()
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    loss.backward()
    opt.step()

    if epoch % LOG_EVERY == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            train_acc = accuracy(logits, y_train)
            test_acc = accuracy(model(X_test), y_test)
            w_norm = weight_norm(model)
        log.append((epoch, train_acc, test_acc, loss.item(), w_norm))
        if epoch % 2000 == 0 or epoch == 1:
            print(f"epoch={epoch:7d} loss={loss.item():.5f} "
                  f"train_acc={train_acc:.4f} test_acc={test_acc:.4f} "
                  f"weight_norm={w_norm:.3f}")

        if test_acc >= GROK_THRESHOLD:
            if grok_checks_remaining is None:
                grok_checks_remaining = GROK_PATIENCE_CHECKS
            grok_checks_remaining -= 1
            if grok_checks_remaining <= 0:
                break

print(f"stopped at epoch {epoch}")

In [ ]:
with open("grokking_results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_accuracy", "test_accuracy", "train_loss", "weight_norm"])
    writer.writerows(log)

epochs, train_accs, test_accs, _, weight_norms = zip(*log)

# lag window: first epoch training memorized the data, to the first epoch
# test accuracy hit the grokking threshold (its de facto maximum)
memorize_epoch = next((e for e, a in zip(epochs, train_accs) if a >= 0.999), None)
grok_epoch = next((e for e, a in zip(epochs, test_accs) if a >= GROK_THRESHOLD), None)

fig, (ax_acc, ax_norm) = plt.subplots(
    2, 1, figsize=(7, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]}
)

ax_acc.plot(epochs, train_accs, color="#0072B2", label="train accuracy")
ax_acc.plot(epochs, test_accs, color="#E69F00", label="test accuracy")
ax_acc.set_ylabel("accuracy")
ax_acc.set_title(f"Grokking: modular addition mod {P}")
ax_acc.legend(loc="upper left")

ax_norm.plot(epochs, weight_norms, color="#555555", label="weight L2 norm")
ax_norm.set_ylabel("weight norm")
ax_norm.set_xlabel("epoch (log scale)")
ax_norm.legend(loc="upper right")

if memorize_epoch is not None and grok_epoch is not None and grok_epoch > memorize_epoch:
    for ax in (ax_acc, ax_norm):
        ax.axvspan(memorize_epoch, grok_epoch, color="#555555", alpha=0.08, zorder=0)
    ax_acc.axvline(memorize_epoch, color="#555555", linestyle="--", linewidth=1)
    ax_acc.axvline(grok_epoch, color="#555555", linestyle="--", linewidth=1)
    ax_acc.annotate("lag", xy=((memorize_epoch * grok_epoch) ** 0.5, 0.5),
                     ha="center", color="#555555")

ax_norm.set_xscale("log")
fig.tight_layout()
fig.savefig("grokking_curve.png", dpi=150)
plt.show()